# Narcolepsy Predictive Model

## Longitudinal feature extraction

In [ ]:
import polars as pl
import pandas as pd
import sys
import os
from pathlib import Path

data_path = Path('data/predictive-modeling') # path to data directory

# please extract features using NarcolepsyModel for all the EHR data for the patients in the cohort
# example code to do this is provided in 'discriminative-modeling/discriminative-example.ipynb', final cell
# the code here uses the features extracted from that notebook

In [2]:
features = pl.read_parquet(data_path / 'features_initial.parquet').drop('index')
diagnosis = pl.read_parquet(data_path / 'diagnosis.parquet')

## create predictive features for NT1 +, -, controls

In [17]:
diagnosis.filter(
    pl.col('diagnosis') == 'NT1',
    pl.col('diagnosis_date').is_not_null()
)

diagnosis,diagnosis_filename,diagnosis_date,comments,bdsp_patient_id,id
str,str,date,str,i64,str
"""NT1""","""Notes_1129913724_446959860_201…",2018-01-12,null,150054658,"""150054658"""
"""NT1""","""12496450.txt""",2014-07-16,null,177545930,"""177545930"""
"""NT1""","""Notes_1154928730_1177834574_20…",2017-02-05,null,175069953,"""175069953"""
"""NT1""","""Notes_13258852030_1054311700_2…",2016-06-02,null,115177066,"""115177066"""
"""NT1""","""Notes_1129894280_464886102_201…",2019-04-26,null,150035401,"""150035401"""
…,…,…,…,…,…
"""NT1""","""Notes_1129912580_468123529_201…",2018-09-17,null,150054128,"""150054128"""
"""NT1""","""Notes_1155130582_1183554306_20…",2018-05-11,null,175271667,"""175271667"""
"""NT1""","""Notes_1129899699_2818581896_20…",2018-04-02,null,150040876,"""150040876"""


In [6]:
os.makedirs(data_path / 'nt1', exist_ok=True) # create directory for NT1 features if it doesn't exist

# features for NT1+ patients
features.join(
    diagnosis.filter(
        pl.col('diagnosis') == 'NT1',
        pl.col('diagnosis_date').is_not_null()
    ).select(['id','diagnosis_date']),
    on='id',
    how='right',
).with_columns(
    pl.when(pl.col('date') < pl.col('diagnosis_date')).then(0).otherwise(1).alias('n+_state'), # labels visits as 0 if they occur before the diagnosis date, and 1 if they occur on or after the diagnosis date
).drop('diagnosis_date').group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt1/n+_features_3.parquet'
)

# features for NT1- patients, who never receive the diagnosis
features.join(
    diagnosis.filter(
        pl.col('diagnosis') == 'Negative',
        (pl.col('comments') != 'CONTROL') | pl.col('comments').is_null() 
    ).select(['id']),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'), # all visits for NT1- patients are labeled as 0, since they never receive the diagnosis
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt1/n-_features_3.parquet'
)

# features for controls
features.join(
    diagnosis.filter(
        pl.col('diagnosis') == 'Negative',
        pl.col('comments') == 'CONTROL'
    ).select(['id']),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'),
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt1/controls_features_3.parquet'
)

## create predictive features for NT2IH +, -, controls

In [8]:
diagnosis.filter(
    pl.col('diagnosis') == 'NT2IH'
)

diagnosis,diagnosis_filename,diagnosis_date,comments,bdsp_patient_id,id
str,str,date,str,i64,str
"""NT2IH""","""Notes_1129899717_416613744_201…",2013-01-02,null,150041043,"""150041043"""
"""NT2IH""","""Note_177519849_129924807_20090…",2009-06-23,null,177519849,"""177519849"""
"""NT2IH""","""13411060.txt""",2017-06-26,null,177540553,"""177540553"""
"""NT2IH""","""15674281.txt""",2019-02-02,null,177518000,"""177518000"""
"""NT2IH""","""I0006_179003931_2743793800_201…",2011-05-08,null,179003931,"""179003931"""
…,…,…,…,…,…
"""NT2IH""","""Notes_13477927426_4509367675_2…",2020-07-19,null,114687567,"""114687567"""
"""NT2IH""","""9211475.txt""",2012-05-07,null,177509985,"""177509985"""
"""NT2IH""","""Notes_1129901401_454842296_201…",2018-01-23,null,150042594,"""150042594"""


In [7]:
os.makedirs(data_path / 'nt2ih', exist_ok=True) # create directory for NT2IH features if it doesn't exist

# features for NT2IH+ patients
features.join(
    diagnosis.filter(
        pl.col('diagnosis') == 'NT2IH',
        pl.col('diagnosis_date').is_not_null()
    ).select(['id','diagnosis_date']),
    on='id',
    how='right',
).with_columns(
    pl.when(pl.col('date') < pl.col('diagnosis_date')).then(0).otherwise(1).alias('n+_state'),
).drop('diagnosis_date').group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt2ih/n+_features_3.parquet'
)

# features for NT2IH- patients, who never receive the diagnosis
features.join(
    diagnosis.filter(
        pl.col('diagnosis') == 'Negative',
        (pl.col('comments') != 'CONTROL') | pl.col('comments').is_null()
    ).select(['id']),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'), # all visits for NT2IH- patients are labeled as 0, since they never receive the diagnosis
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt2ih/n-_features_3.parquet'
)

# features for controls
features.join(
    diagnosis.filter(
        pl.col('diagnosis') == 'Negative',
        pl.col('comments') == 'CONTROL'
    ).select(['id']),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'),
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt2ih/controls_features_3.parquet'
)